In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = '/content/drive/My Drive/Breast Cancer'

# Create necessary subfolders for results, plots, and Grad-CAM views
DATASET_DIR = os.path.join(BASE_DIR, 'Dataset')
RESULTS_DIR = os.path.join(BASE_DIR, 'Result')
PREDICTIONS_DIR = os.path.join(RESULTS_DIR, 'Predictions')
PLOTS_DIR = os.path.join(RESULTS_DIR, 'Overall_Plots')
GRADCAM_DIR = os.path.join(BASE_DIR, 'Grad-Cam view')

for directory in [DATASET_DIR, PREDICTIONS_DIR, PLOTS_DIR, GRADCAM_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Directories successfully created in Google Drive!")


Mounted at /content/drive
Directories successfully created in Google Drive!


In [2]:
!pip install -q ultralytics roboflow grad-cam opencv-python matplotlib seaborn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 658.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.8/336.8 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 kB 10.8 MB/s eta 0:00:00


In [3]:
# !pip install roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key="vj4kwJOzym3JLz5GQJ9A")
# project = rf.workspace("thesis-zhah0").project("breastcancer-yolov8-fsmi8")
# version = project.version(1)
# dataset = version.download("yolo26", location = DATASET_DIR)
# print(f"Datase Successfully downloaded to : {DATASET_DIR}")


In [4]:
import os

print(os.listdir('/content'))

['.config', 'drive', 'sample_data']


In [5]:
# ==========================================
# STEP 3: Train YOLO Model & Save Results
# ==========================================
from ultralytics import YOLO

# Load a pre-trained base model
model = YOLO("yolo26n.pt")

# Train the model (Ultralytics saves runs automatically)
results = model.train(
    data=os.path.join(DATASET_DIR, "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    project=RESULTS_DIR,
    name="training_run"
)

# Path where best weights are saved
best_weights_path = os.path.join(RESULTS_DIR, "training_run", "weights", "best.pt")
print(f"Training complete! Best weights saved at: {best_weights_path}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My Drive/Breast Cancer/Dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=t

In [8]:
import os
import shutil
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Correct path pointing to training_run-2
best_weights_path = "/content/drive/My Drive/Breast Cancer/Result/training_run-2/weights/best.pt"

# Reload trained model
model = YOLO(best_weights_path)

# Run validation to generate performance metrics and evaluation curves
# (Use metrics.seg for segmentation tasks, or metrics.box for bounding boxes)
metrics = model.val(data=os.path.join(DATASET_DIR, "data.yaml"))
print(f"Validation mAP50: {metrics.box.map50:.4f}")

# Move generated evaluation plots to Overall_Plots folder
run_plots_dir = os.path.join(RESULTS_DIR, "training_run-2")
for file_name in os.listdir(run_plots_dir):
    if file_name.endswith('.png'):
        shutil.copy(os.path.join(run_plots_dir, file_name), PLOTS_DIR)

print(f"Evaluation diagrams and plots saved to: {PLOTS_DIR}")

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,375,226 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.9±1.3 ms, read: 13.2±10.4 MB/s, size: 27.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/My Drive/Breast Cancer/Dataset/valid/labels.cache... 164 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 164/164 36.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.9it/s 5.7s
                   all        164        174      0.923      0.877      0.872      0.694
                cancer        116        126       0.98      0.802       0.84      0.659
                normal         48         48      0.867      0.952      0.903      0.728
Speed: 7.2ms preprocess, 8.6ms inference, 

In [9]:
# ==========================================
# STEP 5: Generate Predictions & Visual Outputs
# ==========================================
import cv2
import os
import matplotlib.pyplot as plt

# Pick a sample image from the test set
test_images_path = os.path.join(DATASET_DIR, "test", "images")

if os.path.exists(test_images_path) and len(os.listdir(test_images_path)) > 0:
    sample_images = os.listdir(test_images_path)
    sample_img_name = sample_images[0]
    img_path = os.path.join(test_images_path, sample_img_name)

    # 1. Run standard prediction using the YOLO model (handles boxes/masks automatically)
    pred_results = model(img_path)

    # Plot the results (YOLO automatically draws bounding boxes or segmentation masks)
    res_plotted = pred_results[0].plot()

    # Save to Predictions folder
    pred_output_path = os.path.join(PREDICTIONS_DIR, f"pred_{sample_img_name}")
    cv2.imwrite(pred_output_path, res_plotted)
    print(f"Prediction result successfully saved to: {pred_output_path}")

    # 2. Alternative robust explainability approach for YOLOv8:
    # Instead of manual Grad-CAM hooks which frequently break on YOLO object detection/segmentation architectures,
    # you can save validation batch visual overlays generated directly by YOLO:
    val_results_dir = os.path.join(RESULTS_DIR, "training_run-2")

    # Check for YOLO batch prediction preview images (e.g., val_batch0_pred.jpg)
    found_preview = False
    if os.path.exists(val_results_dir):
        for file_name in os.listdir(val_results_dir):
            if "val_batch" in file_name and file_name.endswith(".jpg"):
                src_preview = os.path.join(val_results_dir, file_name)
                dest_preview = os.path.join(PLOTS_DIR, file_name)
                cv2.imwrite(dest_preview, cv2.imread(src_preview))
                print(f"YOLO validation preview saved to: {dest_preview}")
                found_preview = True

    if not found_preview:
        print("Note: Run model.val() first to generate built-in validation grid visualizations.")

else:
    print(f"Test images directory not found or empty at: {test_images_path}")


image 1/1 /content/drive/My Drive/Breast Cancer/Dataset/test/images/mdb167ll_jpg.rf.4205e7342b9f942c4e3d33e3a08a1171.jpg: 640x640 1 cancer, 10.3ms
Speed: 1.2ms preprocess, 10.3ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
Prediction result successfully saved to: /content/drive/My Drive/Breast Cancer/Result/Predictions/pred_mdb167ll_jpg.rf.4205e7342b9f942c4e3d33e3a08a1171.jpg
YOLO validation preview saved to: /content/drive/My Drive/Breast Cancer/Result/Overall_Plots/val_batch0_labels.jpg
YOLO validation preview saved to: /content/drive/My Drive/Breast Cancer/Result/Overall_Plots/val_batch0_pred.jpg
YOLO validation preview saved to: /content/drive/My Drive/Breast Cancer/Result/Overall_Plots/val_batch1_pred.jpg
YOLO validation preview saved to: /content/drive/My Drive/Breast Cancer/Result/Overall_Plots/val_batch1_labels.jpg
YOLO validation preview saved to: /content/drive/My Drive/Breast Cancer/Result/Overall_Plots/val_batch2_labels.jpg
YOLO validation preview save

In [10]:
# ==========================================
# STEP 5: Generate Model Explainability Heatmaps
# ==========================================
import os
import cv2
from ultralytics import solutions

# Pick a sample image from the test set
test_images_path = os.path.join(DATASET_DIR, "test", "images")

if os.path.exists(test_images_path) and len(os.listdir(test_images_path)) > 0:
    sample_images = os.listdir(test_images_path)
    sample_img_name = sample_images[0]
    img_path = os.path.join(test_images_path, sample_img_name)

    # Initialize the Ultralytics Heatmap module using your trained best weights
    # This acts as your model attention/explainability map for your thesis figures
    heatmap_generator = solutions.Heatmap(
        model=best_weights_path,
        colormap=cv2.COLORMAP_JET,  # Standard Jet colormap for heatmaps
        show=False                  # Do not pop up a window interactively in Colab
    )

    # Read the test image using OpenCV
    im = cv2.imread(img_path)

    if im is not None:
        # Generate the heatmap overlay result
        results = heatmap_generator(im)

        # Save output into your Grad-CAM / Explainability folder
        gradcam_output_path = os.path.join(GRADCAM_DIR, f"heatmap_{sample_img_name}")

        # The processed output frame can be accessed via results.plot_im or similar attributes depending on version,
        # or we can save the plotted frame directly:
        if hasattr(results, "plot_im"):
            cv2.imwrite(gradcam_output_path, results.plot_im)
        else:
            # Fallback wrapper if it returns the image directly
            cv2.imwrite(gradcam_output_path, results)

        print(f"Explainability heatmap successfully saved to: {gradcam_output_path}")
    else:
        print(f"Failed to read image at: {img_path}")

else:
    print(f"Test images directory not found or empty at: {test_images_path}")

Ultralytics Solutions: ✅ {'source': None, 'model': '/content/drive/My Drive/Breast Cancer/Result/training_run-2/weights/best.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 2, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90.0, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'tracktrack.yaml', 'verbose': True, 'data': 'images'}
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 1.08s
Prepared 1 pa

In [11]:
# ==========================================
# STEP 6: Run Predictions & Metrics for ALL Test Images
# ==========================================
import os
import cv2
from ultralytics import YOLO

# 1. Path to test images
test_images_path = os.path.join(DATASET_DIR, "test", "images")

if os.path.exists(test_images_path):
    test_images = [f for f in os.listdir(test_images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(test_images)} test images. Processing predictions...")

    # Loop through every image in the test set
    for img_name in test_images:
        img_path = os.path.join(test_images_path, img_name)

        # Run YOLO prediction
        pred_results = model(img_path, verbose=False)

        # Plot predictions (bounding boxes or segmentation masks)
        res_plotted = pred_results[0].plot()

        # Save output to Predictions folder
        pred_output_path = os.path.join(PREDICTIONS_DIR, f"pred_{img_name}")
        cv2.imwrite(pred_output_path, res_plotted)

    print(f"Successfully generated and saved predictions for all {len(test_images)} test images in: {PREDICTIONS_DIR}")

else:
    print(f"Test directory not found at: {test_images_path}")

# ==========================================
# STEP 7: Compute and Print Overall Test Metrics
# ==========================================
print("\nRunning overall evaluation on the test dataset...")

# Evaluate the model on the test split defined in your data.yaml
test_metrics = model.val(data=os.path.join(DATASET_DIR, "data.yaml"), split='test')

# Print out key metrics for your thesis results chapter
print("\n--- Final Test Evaluation Results ---")
if hasattr(test_metrics, 'seg') and test_metrics.seg is not None:
    print(f"Mask mAP50: {test_metrics.seg.map50:.4f}")
    print(f"Mask mAP50-95: {test_metrics.seg.map:.4f}")
else:
    print(f"Box mAP50: {test_metrics.box.map50:.4f}")
    print(f"Box mAP50-95: {test_metrics.box.map:.4f}")

print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall: {test_metrics.box.mr:.4f}")

Found 493 test images. Processing predictions...
Successfully generated and saved predictions for all 493 test images in: /content/drive/My Drive/Breast Cancer/Result/Predictions

Running overall evaluation on the test dataset...
Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,375,226 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 24.7±9.0 MB/s, size: 28.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/My Drive/Breast Cancer/Dataset/test/labels... 493 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 493/493 1.7it/s 4:51
val: New cache created: /content/drive/My Drive/Breast Cancer/Dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 3.8it/s 8.1s
 

In [12]:
# ==========================================
# STEP 8: Generate Multiple Explainability Views (5-10 Test Images)
# ==========================================
import os
import cv2
from ultralytics import solutions

test_images_path = os.path.join(DATASET_DIR, "test", "images")

if os.path.exists(test_images_path):
    test_images = [f for f in os.listdir(test_images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Select the first 5 to 10 images (e.g., up to 10)
    num_images_to_process = min(10, len(test_images))
    selected_images = test_images[:num_images_to_process]

    print(f"Generating explainability heatmaps for {num_images_to_process} test images...")

    for img_name in selected_images:
        img_path = os.path.join(test_images_path, img_name)

        # Initialize the heatmap generator for each image
        heatmap_generator = solutions.Heatmap(
            model=best_weights_path,
            colormap=cv2.COLORMAP_JET,
            show=False
        )

        im = cv2.imread(img_path)
        if im is not None:
            results = heatmap_generator(im)

            # Define output path in your Grad-Cam view folder
            gradcam_output_path = os.path.join(GRADCAM_DIR, f"heatmap_{img_name}")

            # Save the plotted frame
            if hasattr(results, "plot_im"):
                cv2.imwrite(gradcam_output_path, results.plot_im)
            else:
                cv2.imwrite(gradcam_output_path, results)

            print(f"Saved: {gradcam_output_path}")

    print(f"Successfully generated all explainability views in: {GRADCAM_DIR}")
else:
    print(f"Test directory not found at: {test_images_path}")

Generating explainability heatmaps for 10 test images...
Ultralytics Solutions: ✅ {'source': None, 'model': '/content/drive/My Drive/Breast Cancer/Result/training_run-2/weights/best.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 2, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90.0, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'tracktrack.yaml', 'verbose': True, 'data': 'images'}
0: 640x640 4.6ms, 1 cancer
Speed: 293.9ms track, 4.6ms solution per image at shape (1, 3, 640, 640)

Saved: /content/drive/My Drive/Breast Cancer/Grad-Cam view/heatmap_mdb167ll_jpg.rf.4205e73